In [ ]:
from flask import Flask, jsonify, request
import requests
import threading

app = Flask(__name__)



NASA_API_KEY = "DEMO_KEY"
NASA_APOD_URL = "https://api.nasa.gov/planetary/apod"
NASA_NEO_URL  = "https://api.nasa.gov/neo/rest/v1/feed"
NASA_MARS_URL = "https://api.nasa.gov/mars-photos/api/v1/rovers/curiosity/photos"




@app.route("/")

def home():
    return jsonify({
        "message": "Welcome to NASA App",
        "endpoints": [
            "/apod            - Astronomy Picture of the Day",
            "/apod?date=YYYY-MM-DD - APOD for a specific date",
            "/neo             - Near Earth Objects (today)",
            "/mars            - Mars Rover Photos"
        ]
    })



@app.route("/apod")
def apod():
    date = request.args.get("date", "")
    params = {"api_key": NASA_API_KEY}
    if date:
        params["date"] = date
    response = requests.get(NASA_APOD_URL, params=params)
    data = response.json()
    return jsonify({
        "title":       data.get("title"),
        "date":        data.get("date"),
        "explanation": data.get("explanation"),
        "url":         data.get("url"),
        "media_type":  data.get("media_type")
    })



@app.route("/neo")
def neo():
    from datetime import date
    today = str(date.today())
    params = {
        "start_date": today,
        "end_date":   today,
        "api_key":    NASA_API_KEY
    }
    response = requests.get(NASA_NEO_URL, params=params)
    data = response.json()
    asteroids = data.get("near_earth_objects", {}).get(today, [])
    result = []
    for asteroid in asteroids[:5]:
        result.append({
            "name":             asteroid["name"],
            "diameter_km":      asteroid["estimated_diameter"]["kilometers"]["estimated_diameter_max"],
            "is_hazardous":     asteroid["is_potentially_hazardous_asteroid"],
            "close_approach":   asteroid["close_approach_data"][0]["close_approach_date"],
            "miss_distance_km": asteroid["close_approach_data"][0]["miss_distance"]["kilometers"]
        })
    return jsonify({"date": today, "asteroids": result})



@app.route("/mars")

def mars():
    params = {
        "sol":     1000,
        "api_key": NASA_API_KEY
    }
    response = requests.get(NASA_MARS_URL, params=params)
    data = response.json()
    photos = data.get("photos", [])[:5]
    result = []
    for photo in photos:
        result.append({
            "id":         photo["id"],
            "sol":        photo["sol"],
            "camera":     photo["camera"]["full_name"],
            "earth_date": photo["earth_date"],
            "image_url":  photo["img_src"]
        })
    return jsonify({"rover": "Curiosity", "photos": result})


def run_app():
    app.run(debug=False, port=5000, use_reloader=False)


thread = threading.Thread(target=run_app)
thread.daemon = True
thread.start()



print("Flask server is running at: http://127.0.0.1:5000")
print("Open these URLs in your browser:")
print("  http://127.0.0.1:5000/")
print("  http://127.0.0.1:5000/apod")
print("  http://127.0.0.1:5000/neo")
print("  http://127.0.0.1:5000/mars")


Flask server is running at: http://127.0.0.1:5000
Open these URLs in your browser:
  http://127.0.0.1:5000/
  http://127.0.0.1:5000/apod
  http://127.0.0.1:5000/neo
  http://127.0.0.1:5000/mars


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
